# Métricas de Fairness en el Modelo Predictivo

Este documento presenta un análisis de las métricas de fairness utilizadas para evaluar nuestro modelo predictivo. Este modelo tiene como objetivo poner a prueba nuestras hipotesis al buscar comparar estas metricas usando las justificacion de la violencia en casa como nuestra variable con el atributo sensible. Las métricas de fairness aseguran que el modelo trate equitativamente a todos los grupos involucrados.

## Autores

- **Patricio Villanueva Gio**
- **Dafne Tamayo Leon**
- **Maria Paula Perez Romo**

## Descripción de Métricas de Fairness

### Paridad Demográfica

- **¿Qué mide?**: Evalúa si la probabilidad de recibir una predicción positiva es igual entre los grupos, asegurando que ningún grupo demográfico sea sistemáticamente favorecido o perjudicado.
- **Cómo se calcula**: `P(Y_pred = 1 | G = grupo1) = P(Y_pred = 1 | G = grupo2)`
- **Interpretación**: Un valor cercano entre grupos indica que el modelo ofrece resultados equitativos sin favorecer a un grupo sobre otro.

### Paridad Predictiva

- **¿Qué mide?**: Asegura que los valores predictivos positivos sean iguales entre los grupos.
- **Cómo se calcula**: `P(Y = 1 | Y_pred = 1, G = grupo1) = P(Y = 1 | Y_pred = 1, G = grupo2)`
- **Interpretación**: Indica que las predicciones positivas del modelo tienen la misma precisión en todos los grupos.

### Igualdad de Oportunidades

- **¿Qué mide?**: Garantiza que la tasa de verdaderos positivos sea la misma para todos los grupos.
- **Cómo se calcula**: `P(Y_pred = 1 | Y = 1, G = grupo1) = P(Y_pred = 1 | Y = 1, G = grupo2)`
- **Interpretación**: Cada grupo tiene la misma probabilidad de ser correctamente identificado como positivo por el modelo.

### Equidad de Tasa de Falsos Positivos

- **¿Qué mide?**: Evalúa si un modelo de clasificación trata de manera equitativa a diferentes grupos en términos de asignar predicciones positivas incorrectas.
- **Cómo se calcula**: `P(Y_pred = 1 | Y = 0, G = grupo1) = P(Y_pred = 1 | Y = 0, G = grupo2)`
- **Interpretación**: Asegura que los grupos no tengan una probabilidad desproporcionadamente alta de recibir falsas alarmas.

### Disparate Impact

- **¿Qué mide?**: Evalúa las proporciones de predicciones positivas entre grupos para identificar sesgos no intencionados.
- **Cómo se calcula**: `P(Y_pred = 1 | G = grupo1) / P(Y_pred = 1 | G = grupo2)`
- **Interpretación**: Un valor superior a 1 indica un posible favoritismo hacia el grupo 1, mientras que un valor inferior a 1 podría indicar un sesgo contra el mismo.

## Conclusiones

Conclusiones del Análisis de Fairness

- Sesgos en la Asignación de Predicciones:
    - Paridad Demográfica y Disparate Impact muestran claramente que hay un sesgo en cómo el modelo asigna predicciones positivas. El Grupo 0 recibe predicciones positivas con mayor frecuencia en comparación con el Grupo 1, lo que sugiere un sesgo en la asignación que puede necesitar ser ajustado para evitar la discriminación y promover un tratamiento más equitativo entre los grupos.

- Diferencias en la Precisión de las Predicciones:
    - Paridad Predictiva, Igualdad de Precisión, y Equidad de Tasa de Falsos Positivos indican que el modelo es más preciso para el Grupo 0 que para el Grupo 1. Esto podría ser problemático si el modelo se utiliza en un contexto donde decisiones críticas dependen de estas predicciones, ya que el Grupo 1 está en desventaja debido a una mayor tasa de clasificaciones incorrectas.

- Tratamiento de Verdaderos Positivos y Negativos:
    - Igualdad de Oportunidades y Paridad de Tasa de Falsos Negativos revelan que el Grupo 0 tiene una mejor probabilidad de ser identificado correctamente como positivo, y menos probabilidades de ser mal clasificado como negativo en comparación con el Grupo 1. Esto destaca una falta de equidad en cómo el modelo identifica y clasifica casos verdaderos, lo cual es crucial para mantener la confianza en las aplicaciones prácticas del modelo.

- Tasa General de Errores:
    - Paridad de Tasa de Error muestra que el Grupo 1 tiene una mayor tasa de error en general, lo que refuerza la necesidad de mejorar la precisión del modelo para este grupo para asegurar que las decisiones basadas en el modelo sean justas y confiables para todos los usuarios.

---

Este análisis es crucial para desarrollar modelos más justos y éticos, particularmente en aplicaciones que afectan significativamente a las vidas de las personas.


In [71]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/Process/DataSetAnalisis.csv").set_index("year")

In [72]:
import pandas as pd

def calcular_metricas_fairness(df, etiqueta_verdadera, prediccion, grupo):
    df['TP'] = (df[prediccion] == 1) & (df[etiqueta_verdadera] == 1)
    df['FP'] = (df[prediccion] == 1) & (df[etiqueta_verdadera] == 0)
    df['TN'] = (df[prediccion] == 0) & (df[etiqueta_verdadera] == 0)
    df['FN'] = (df[prediccion] == 0) & (df[etiqueta_verdadera] == 1)

    resultados = df.groupby(grupo).apply(lambda x: pd.Series({
        'Paridad Demográfica': x[prediccion].mean(),
        'Paridad Predictiva': x['FP'].sum() / (x['FP'].sum() + x['TN'].sum()) if (x['FP'].sum() + x['TN'].sum()) > 0 else 0,
        'Igualdad de Oportunidades': x['TP'].sum() / (x['TP'].sum() + x['FN'].sum()) if (x['TP'].sum() + x['FN'].sum()) > 0 else 0,
        'Equidad de Tasa de Falsos Positivos': x['FP'].sum() / (x['FP'].sum() + x['TN'].sum()) if (x['FP'].sum() + x['TN'].sum()) > 0 else 0,
        'Paridad de Tasa de Falsos Negativos': x['FN'].sum() / (x['FN'].sum() + x['TP'].sum()) if (x['FN'].sum() + x['TP'].sum()) > 0 else 0,
        'Paridad de Tasa de Error': (x['FP'].sum() + x['FN'].sum()) / (x['TP'].sum() + x['TN'].sum() + x['FP'].sum() + x['FN'].sum()),
        'Igualdad de Precisión': (x['TP'].sum() + x['TN'].sum()) / (x['TP'].sum() + x['TN'].sum() + x['FP'].sum() + x['FN'].sum()),
        'Disparate Impact': (x[x[prediccion] == 1].shape[0] / x.shape[0]) / (df[df[prediccion] == 1].shape[0] / df.shape[0]) if df[df[prediccion] == 1].shape[0] > 0 else 0,
    }))
    return resultados

metricas_fairness = calcular_metricas_fairness(df, 'Happiness', 'HappinessPred', 'justificacion_violencia')
metricas_fairness


/tmp/ipykernel_784221/829761714.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resultados = df.groupby(grupo).apply(lambda x: pd.Series({


,Paridad Demográfica,Paridad Predictiva,Igualdad de Oportunidades,Equidad de Tasa de Falsos Positivos,Paridad de Tasa de Falsos Negativos,Paridad de Tasa de Error,Igualdad de Precisión,Disparate Impact
justificacion_violencia,,,,,,,,
0,0.645161,0.571429,0.705882,0.571429,0.294118,0.419355,0.580645,1.146953
1,0.484848,0.470588,0.500000,0.470588,0.500000,0.484848,0.515152,0.861953


## Interpretación de las metricas

- Paridad Demográfica
    - Grupo 0: 0.645161
    - Grupo 1: 0.484848
    - __Interpretación: Hay una mayor probabilidad de que los individuos en el grupo 0 reciban una predicción positiva en comparación con el grupo 1. Esto podría indicar un sesgo en cómo el modelo asigna predicciones a los diferentes grupos.__

- Paridad Predictiva
    - Grupo 0: 0.571429
    - Grupo 1: 0.470588
    - __Interpretación: El modelo tiene diferentes tasas de precisión en sus predicciones positivas entre los grupos. El grupo 0 tiene una mayor proporción de falsos positivos en relación con los verdaderos negativos comparado con el grupo 1.__

- Igualdad de Oportunidades
    - Grupo 0: 0.705882
    - Grupo 1: 0.500000
    - __Interpretación: El grupo 0 tiene más probabilidades de ser correctamente identificado como positivo (si realmente son positivos) en comparación con el grupo 1. Esta es una métrica importante para asegurar que el modelo no sea más favorable a un grupo sobre otro en términos de capturar los verdaderos positivos.__

- Equidad de Tasa de Falsos Positivos
    - Grupo 0: 0.571429
    - Grupo 1: 0.470588
    - __Interpretación: Similar a la Paridad Predictiva, muestra la frecuencia con la que los individuos que no deberían ser positivos son incorrectamente etiquetados como tales. El grupo 0 tiene una tasa más alta que el grupo 1.__

- Paridad de Tasa de Falsos Negativos

    - Grupo 0: 0.294118
    - Grupo 1: 0.500000
    - __Interpretación: El grupo 0 tiene una menor probabilidad de que los verdaderos positivos sean incorrectamente etiquetados como negativos en comparación con el grupo 1.__

- Paridad de Tasa de Error
    - Grupo 0: 0.419355
    - Grupo 1: 0.484848
    - __Interpretación: Esta métrica muestra la tasa de todas las predicciones incorrectas. El grupo 1 tiene una tasa de error general más alta, lo que podría indicar un peor rendimiento del modelo para este grupo.__

- Igualdad de Precisión

    - Grupo 0: 0.580645
    - Grupo 1: 0.515152
    - __Interpretación: La precisión global del modelo es más baja para el grupo 1, lo que indica que el grupo 0 está recibiendo predicciones más precisas en general.__

- Disparate Impact
    - Grupo 0: 1.146953
    - Grupo 1: 0.861953
    - __Interpretación: Un disparate impact mayor que 1 para el grupo 0 sugiere que este grupo es más probable que reciba predicciones positivas en comparación con la "tasa base" del grupo 1. Esto puede ser un indicador de sesgo favorable hacia el grupo 0.__